# fhkit.at: what the browser shows

`fhkit-public-site.ipynb` reads the built HTML. This one renders it in
Chromium, served the way Cloudflare Pages serves it, because the worst
defects of 2026-10-10 left the markup looking fine: React threw hydration
errors #418 and #423 on every home page and rendered it a second time, dark
mode put black text and icons on navy, and the bar's Sign In threw "the OIDC
runtime is not loaded on this route" instead of navigating.

Every page of the sitemap is loaded on desktop and on a phone, light and dark,
then unknown addresses, signing in from the bar and from the phone menu, the
map notice, the partner cards, and pressing Send in the contact dialog.

Needs the Python package `playwright`, the system Chromium and a finished
`gatsby build`. The probe, `fhkit_browser_probe.py`, runs as a subprocess,
because the kernel already runs an asyncio loop. Nothing leaves the machine:
it serves `public/` itself and aborts every request to another host, after
writing down what was asked for. The contact form's POST to emailwerk is
answered inside the browser, so no mail is sent. A run takes about six
minutes.

In [ ]:
import json, os, re, subprocess, sys
from pathlib import Path

SITE = Path(os.environ.get('FHKIT_SITE_DIR', '..')).resolve()
LOCALES = ['de', 'en', 'sl', 'it', 'ja']
PREFIX = {loc: '' if loc == 'de' else f'{loc}/' for loc in LOCALES}

RESULTS = []

def check(name, ok, detail=''):
    RESULTS.append({'name': name, 'pass': bool(ok), 'detail': str(detail)})
    print(('PASS  ' if ok else 'FAIL  ') + name + (f'  ({detail})' if detail else ''))
    return bool(ok)

probe = subprocess.run([sys.executable, str(SITE / 'tests' / 'fhkit_browser_probe.py'), str(SITE)],
                       capture_output=True, text=True, timeout=1800)
check('the browser probe ran', probe.returncode == 0, probe.stderr.strip()[-400:])
assert probe.returncode == 0, 'the probe died:\n' + probe.stderr[-2000:]
SEEN = json.loads(probe.stdout)
BASE = SEEN['base']
print(f"{len(SEEN['pages'])} page loads and {len(SEEN['not_found'])} unknown addresses, served at {BASE}")

def own_errors(errors, not_found=False):
    """The errors the site caused itself.

    An aborted request to another host reports "Failed to load resource" from
    that host's address; that is the probe, not the site. On a 404 page the
    document and its page data answer 404 on purpose."""
    def expected(e):
        if not e['text'].startswith('Failed to load resource'):
            return False
        return not e['url'].startswith(BASE) or (not_found and 'status of 404' in e['text'])
    return [e['text'] for e in errors if not expected(e)]

def load(path, viewport, mode):
    return next(p for p in SEEN['pages'] if (p['path'], p['viewport'], p['mode']) == (path, viewport, mode))

## Every page renders cleanly

Status 200, nothing thrown or logged as an error, React's hydration errors
included, no picture of the site's own failing, nothing pushing the page
sideways, and no host asked for but osg.snek.at, where the partner photos
are.

In [ ]:
for p in SEEN['pages']:
    errors = own_errors(p['errors'])
    check(f"{p['path']} {p['viewport']} {p['mode']}: 200 and no error in the browser",
          p['status'] == 200 and not errors, errors[:2] or p['status'])

hydration = sorted({f"{p['path']} {p.get('viewport', 'desktop')} {p['mode']}" for p in SEEN['pages'] + SEEN['not_found']
                    for e in p['errors'] if re.search(r'#(418|423|425)\b|[Hh]ydrat', e['text'])})
check('no page throws a React hydration error (#418, #423, #425)', not hydration, hydration[:5])

broken = sorted({f"{p['path']}: {i['src']}" for p in SEEN['pages'] for i in p['images']
                 if i['broken'] and i['shown'] and i['src'].startswith(BASE)})
check("none of the site's own pictures fails to load", not broken, broken[:5])

sideways = sorted({f"{p['path']} {p['viewport']}: {p['sideways']}px" for p in SEEN['pages'] if p['sideways'] > 1})
check('no page scrolls sideways, desktop or phone', not sideways, sideways[:5])

hosts = sorted({u for p in SEEN['pages'] + SEEN['not_found'] for u in p['outside']
                if not u.startswith('https://osg.snek.at/')})
check('no host is asked for but osg.snek.at (the partner photos)', not hosts, hosts[:5])

## Dark mode is readable

jaen keeps the colour mode in localStorage, so whoever switched the CMS frame
to dark sees the public pages dark as well. Every visible text and every icon
in a link or button is measured against the backgrounds behind it, with the
opacity of the element and its ancestors applied. Nothing may fall below 3:1.
The one exception is the footer's copyright line, which the design keeps at
20% opacity until it is hovered, in both modes.

The parts that disappeared on navy are named on their own and must reach
4.5:1 in both modes: the hero heading, the two contact buttons, the five
icons under the photo and the bar's current page.

In [ ]:
FAINT = 'Copyright ©'

for p in SEEN['pages']:
    where = f"{p['path']} {p['viewport']} {p['mode']}"
    if p['mode'] == 'dark':
        check(f'{where}: the page is dark', p['dark'])
    low = [f"{c['text']!r} {c['ratio']}" for c in p['low_contrast'] if not c['text'].startswith(FAINT)]
    check(f'{where}: no text or icon below 3:1', not low, low[:3])

HERO = {'de': 'Lass dich von mir beraten', 'en': 'Let me advise you', 'sl': 'Naj ti svetujem',
        'it': 'Lasciati consigliare da me', 'ja': '私にご相談ください'}
CONTACT_ME = {'de': 'Kontaktiere mich', 'en': 'Contact me', 'sl': 'Kontaktiraj me', 'it': 'Contattami',
              'ja': 'お問い合わせ'}
REQUEST = {'de': 'Jetzt anfragen', 'en': 'Get in touch', 'sl': 'Pišite nam', 'it': 'Richiedi ora',
           'ja': 'お問い合わせはこちら'}
IMPRINT = {'de': 'Impressum', 'en': 'Legal notice', 'sl': 'Impresum', 'it': 'Note legali', 'ja': '法的表示'}
ICONS = ['Netsnek', 'Github', 'Linkedin', 'Instagram', 'Telegram']

def ratios(page, text, kind, nav=None):
    return [c['ratio'] for c in page['contrast']
            if c['text'] == text and c['kind'] == kind and (nav is None or c['nav'] == nav)]

for loc in LOCALES:
    for mode in ['light', 'dark']:
        home = load(f'/{PREFIX[loc]}', 'desktop', mode)
        named = [('hero heading', HERO[loc], 'text'), ('contact button', CONTACT_ME[loc], 'text'),
                 ('bar button', REQUEST[loc], 'text')] + [(f'{icon} icon', icon, 'icon') for icon in ICONS]
        for label, text, kind in named:
            r = ratios(home, text, kind)
            check(f'{loc} {mode}: the {label} reaches 4.5:1', r and min(r) >= 4.5, r)
        notice = load(f'/{PREFIX[loc]}imprint/', 'desktop', mode)
        r = ratios(notice, IMPRINT[loc], 'text', nav=True)
        check(f"{loc} {mode}: the bar's current page reaches 4.5:1", r and min(r) >= 4.5, r)

## Unknown addresses

Pages answers with the nearest 404.html and status 404. The page speaks the
language of the address, keeps the address the visitor typed, is marked
noindex and links home in that language. /docs/ and /experiments/ are among
them now that the placeholders are gone.

In [ ]:
NOT_FOUND = {'de': 'Seite nicht gefunden', 'en': 'Page not found', 'sl': 'Strani ni mogoče najti',
             'it': 'Pagina non trovata', 'ja': 'ページが見つかりません'}

for n in SEEN['not_found']:
    loc, where = n['locale'], f"{n['address']} ({n['mode']})"
    check(f'{where}: status 404', n['status'] == 404, n['status'])
    check(f'{where}: language, heading and title in {loc}',
          n['lang'] == loc and (n['h1'] or '').strip() == NOT_FOUND[loc] and n['title'] == NOT_FOUND[loc],
          f"{n['lang']} {n['h1']!r} {n['title']!r}")
    check(f'{where}: the address stays as typed', n['path'] == n['address'], n['path'])
    check(f'{where}: noindex and a link to /{PREFIX[loc]}',
          n['robots'] == 'noindex' and n['home_link'] == f'/{PREFIX[loc]}', f"{n['robots']} {n['home_link']}")
    errors = own_errors(n['errors'], not_found=True)
    check(f'{where}: no error in the browser', not errors, errors[:2])

## Signing in

The bar's entry is a plain link to /login/, loaded in full, and the sign-in
page then reaches for accounts.netsnek.com (the probe stops it there).
Nothing may be thrown on the way, least of all "the OIDC runtime is not
loaded on this route". What jaen's own sign-in page logs afterwards is not
judged here: its footer links /privacy-policy, /terms-of-service, /contact
and /about, which this site does not have.

The phone menu lists the site's pages, sign-in and Netsnek, in the page's
language. In dark mode nothing in it falls below 3:1. In light mode its two
section titles do, at 2.63:1: they are the template's orange #f77f00 on
white, as on the live site, and stay a known gap until the colour is
changed. Nothing else may.

In [ ]:
def thrown(errors):
    return [e['text'] for e in errors if e['kind'] == 'exception' or 'OIDC' in e['text']]

for loc, s in SEEN['sign_in'].items():
    check(f"{loc}: the bar's sign-in is a plain link to /login/", s['href'] == '/login/' and not s['target'],
          f"{s['href']} {s['target']}")
    check(f'{loc}: a click loads /login/, which reaches for accounts.netsnek.com',
          s['url_after'] == '/login/' and s['accounts_netsnek'], s.get('probe_errors') or s['url_after'])
    errors = own_errors(s['errors']) + thrown(s['after'])
    check(f'{loc}: nothing is thrown on the way', not errors, errors[:2])

DRAWER = {'de': [['Startseite', '/'], ['Impressum', '/imprint/'], ['Anmelden', '/login/'],
                 ['Netsnek', 'https://netsnek.com']],
          'ja': [['ホーム', '/ja/'], ['法的表示', '/ja/imprint/'], ['ログイン', '/login/'],
                 ['Netsnek', 'https://netsnek.com']]}
MENU = {'de': 'Menü', 'ja': 'メニュー'}
ORANGE_TITLES = {'de': ['Navigation', 'Mehr'], 'ja': ['ナビゲーション', 'その他']}

for key, d in SEEN['drawer'].items():
    loc, mode = key.split()
    check(f'{loc} phone menu, {mode}: named and open', d['label'] == MENU[loc] and d['expanded'] == 'true',
          f"{d['label']} {d['expanded']}")
    check(f'{loc} phone menu, {mode}: home, legal notice, sign-in and Netsnek', d['items'] == DRAWER[loc], d['items'])
    known = ORANGE_TITLES[loc] if mode == 'light' else []
    low = [f"{c['text']!r} {c['ratio']}" for c in d['contrast']
           if not c['text'].startswith(FAINT) and not (c['text'] in known and c['ratio'] >= 2.5)]
    check(f'{loc} phone menu, {mode}: nothing below 3:1' + (' but the two orange titles' if known else ''),
          not low, low[:3])
    if mode == 'light':
        check(f'{loc} phone menu: sign-in loads /login/ and reaches for accounts.netsnek.com',
              d['url_after'] == '/login/' and d['accounts_netsnek'], d.get('probe_errors') or d['url_after'])
    errors = own_errors(d['errors']) + thrown(d.get('after', []))
    check(f'{loc} phone menu, {mode}: nothing is thrown', not errors, errors[:2])

## Map notice, contact dialog, partner cards

Until analytics cookies are allowed the office map is replaced by a notice,
which used to say "Google Maps is nicht verfügbar" in every language. No
request goes to Google before consent. The contact button still opens its
dialog. The three partner cards share the row from the md breakpoint up; the
Kanbon card was squeezed to about 165px.

In [ ]:
MAPS = {'de': 'Google Maps ist nicht verfügbar.', 'en': 'Google Maps is unavailable.',
        'sl': 'Google Zemljevidi niso na voljo.', 'it': 'Google Maps non è disponibile.',
        'ja': 'Google マップを利用できません。'}

for loc, m in SEEN['maps'].items():
    check(f'{loc}: the map notice is in {loc}', any(n.startswith(MAPS[loc]) for n in m['notice']), m['notice'])
    check(f'{loc}: no request to Google before consent', not m['google'], m['google'][:2])
    check(f'{loc}: the contact button opens the dialog', m['dialog'] == 1, m.get('probe_errors') or m['dialog'])
    errors = own_errors(m['errors'])
    check(f'{loc}: nothing is thrown', not errors, errors[:2])

for width, cards in SEEN['cards'].items():
    widths = [w for _, w in cards]
    check(f'{width}px: the three partner cards share the row equally',
          len(widths) == 3 and max(widths) - min(widths) <= 1 and min(widths) >= 200, cards)

## Pressing Send

The contact form posts to emailwerk, to fhkit's own public template "fhkit
Contact". The probe fills it in and presses Send in every language. It
writes down every POST to emailwerk.com/graphql and then answers it inside
the browser, so no mail is sent: it aborts the request as if the network
failed, refuses it the way emailwerk refuses (an error with a code), fails it
with an HTTP error (401 as emailwerk's own auth answers, and 500), or accepts
it the way emailwerk queues a mail.

The request names the template and sends the form fields as typed. The
mutation selects `sendTemplateMail` and nothing beside it, which is all
emailwerk runs without credentials. The visitor is only the reply-to, because
emailwerk takes the recipient from the template. Of the confirmation mail only
the page's language travels along: its subject and texts are fixed in the
template, one set per language, so the request carries `locale` and no `t_`
text. It carries no credentials, so emailwerk takes its public branch.

One press makes exactly one POST, whatever emailwerk answers, also when the
network fails and also on a double click. A send is never repeated, because
the network can fail after emailwerk has accepted the send, and a repeat
would deliver the enquiry and the visitor's confirmation a second time. That
is why the form posts with a plain fetch and not through jaen's emailwerk
plugin, whose GraphQL client, gqty, repeated a network failure three times
one second apart.

The outcome shows in the dialog, in the page's language, and takes the focus.
A sent message replaces the form with a thank-you and a Close button. A
failed one keeps the form as typed and offers the enquiry as a prepared mail
to office@fhkit.at. That notice sits at the top of the form, because on a
phone jaen's cookie banner covers the bottom of the dialog until it is
answered. The notice's text reaches 4.5:1 in both modes, and nothing else in
the dialog falls below 3:1.

In [ ]:
from urllib.parse import parse_qs, urlsplit

TEMPLATE = 'cmv2fl2cu14ebps2p05nwpoev'  # "fhkit Contact" in emailwerk
EMAILWERK = 'https://emailwerk.com/graphql'
FORM = {'firstName': 'Erika', 'lastName': 'Musterfrau', 'email': 'erika.musterfrau@example.org',
        'phone': '+43 1 234 56 78', 'message': 'Zeile eins\nZeile <b>zwei</b>'}
HEADING = {'de': 'Kontaktanfrage', 'en': 'Contact enquiry', 'sl': 'Povpraševanje',
           'it': 'Richiesta di contatto', 'ja': 'お問い合わせ'}
SENT = {'de': 'Ihre Nachricht wurde gesendet', 'en': 'Your message has been sent',
        'sl': 'Vaše sporočilo je bilo poslano', 'it': 'Il tuo messaggio è stato inviato',
        'ja': 'メッセージを送信しました'}
NOT_SENT = {'de': 'Ihre Nachricht konnte nicht gesendet werden', 'en': 'Your message could not be sent',
            'sl': 'Vašega sporočila ni bilo mogoče poslati', 'it': 'Non è stato possibile inviare il tuo messaggio',
            'ja': 'メッセージを送信できませんでした'}
CLOSE = {'de': 'Schließen', 'en': 'Close', 'sl': 'Zapri', 'it': 'Chiudi', 'ja': '閉じる'}

# The root of the operation selects sendTemplateMail and nothing else, the
# one shape emailwerk's anonymous gate lets through without credentials.
ALONE = re.compile(r'mutation\b[^{]*\{\s*(?:\w+\s*:\s*)?sendTemplateMail\s*\([^)]*\)\s*\{[^{}]*\}\s*\}')

def request(post):
    body = json.loads(post['body'])
    (args,) = body['variables'].values()
    return body['query'], args

for c in SEEN['contact']:
    loc, posts = c['locale'], c['posts']
    where = (f"{loc} {c['viewport']} {c['mode']}, {c['answer']}" + ('' if c['consent'] else ', no consent')
             + (', double click' if c['double'] else ''))
    dialog = c['dialog'] or {}
    notice = dialog.get('notice') or {}
    errors = own_errors(c['errors'])
    check(f'{where}: nothing is thrown or logged', not errors and not c.get('probe_errors'),
          (c.get('probe_errors') or errors)[:2])
    hosts = [u for u in c['outside'] if not u.startswith('https://osg.snek.at/')]
    check(f'{where}: no other host is asked for but osg.snek.at', not hosts, hosts[:3])

    if not c['consent']:
        check(f'{where}: nothing is posted without the consent box ticked', not posts, len(posts))
        continue

    first = posts[0] if posts else None
    check(f'{where}: Send posts to {EMAILWERK}', first and first['method'] == 'POST' and first['url'] == EMAILWERK,
          first and f"{first['method']} {first['url']}")
    if not first:
        continue
    query, args = request(first)
    values = args.get('values') or {}
    check(f"{where}: the mutation is sendTemplateMail alone, with fhkit's template",
          ALONE.fullmatch(query.strip()) and args.get('templateId') == TEMPLATE,
          f"{args.get('templateId')} {query}")
    check(f'{where}: no recipient is sent, the visitor is only the reply-to',
          'to' not in args and args.get('envelopeOverride') == {'replyTo': FORM['email']},
          {k: v for k, v in args.items() if k not in ('values', 'templateId')})
    check(f'{where}: the form fields go as typed', all(values.get(k) == v for k, v in FORM.items()),
          {k: values.get(k) for k in FORM})
    check(f'{where}: the page address and the language go along',
          values.get('invokedOnUrl') == c['page_url'] and values.get('locale') == loc,
          f"{values.get('invokedOnUrl')} {values.get('locale')}")
    texts = sorted(k for k in values if k.startswith('t_'))
    check(f'{where}: no text of the confirmation mail goes along, only the fields, the page and the language',
          not texts and sorted(values) == sorted(list(FORM) + ['invokedOnUrl', 'locale']),
          texts or sorted(values))
    check(f'{where}: sent as JSON without credentials',
          first['headers'].get('content-type') == 'application/json'
          and not {'authorization', 'cookie'} & set(first['headers']), first['headers'])

    at = [p['at'] for p in posts]
    check(f'{where}: exactly one POST', len(posts) == 1, f'{len(posts)} POSTs at {at} s')

    if c['answer'] == 'accept':
        check(f'{where}: the dialog says the message was sent, in {loc}',
              notice.get('role') == 'status' and notice.get('text', '').startswith(SENT[loc]), notice.get('text'))
        check(f"{where}: the notice has the focus and the only labelled button is {CLOSE[loc]!r}",
              notice.get('focused') and dialog.get('buttons') == [CLOSE[loc]],
              f"{notice.get('focused')} {dialog.get('buttons')}")
        if 'after_close' in c:
            again = c.get('reopened') or {}
            check(f'{where}: Close closes the dialog, and it opens again with an empty form',
                  c['after_close'] is None and again.get('notice') is None and again.get('fields')
                  and not any(again['fields'].values()), f"{c['after_close']} {again}")
    else:
        check(f'{where}: the dialog says the message was not sent, in {loc}',
              notice.get('role') == 'alert' and notice.get('text', '').startswith(NOT_SENT[loc])
              and 'office@fhkit.at' in notice.get('text', ''), notice.get('text'))
        check(f'{where}: the notice has the focus and the form keeps what was typed',
              notice.get('focused') and dialog.get('fields') == FORM, f"{notice.get('focused')} {dialog.get('fields')}")
        link = urlsplit(notice.get('mailto') or '')
        mail = {k: v[0] for k, v in parse_qs(link.query, keep_blank_values=True).items()}
        body = mail.get('body', '')
        check(f'{where}: the notice offers the enquiry as a mail to office@fhkit.at',
              link.scheme == 'mailto' and link.path == 'office@fhkit.at'
              and mail.get('subject') == f"{HEADING[loc]}: {FORM['firstName']} {FORM['lastName']}"
              and all(x in body for x in [FORM['email'], FORM['phone'], 'Zeile eins\r\nZeile <b>zwei</b>'])
              and '\n' not in body.replace('\r\n', ''), notice.get('mailto'))
        if dialog.get('banner'):
            check(f"{where}: the notice is not behind jaen's cookie banner",
                  notice['box']['bottom'] <= dialog['banner']['top'], f"{notice['box']} {dialog['banner']}")
    ratios = sorted({x['ratio'] for x in c['notice_contrast']})
    check(f"{where}: the notice's text reaches 4.5:1", ratios and ratios[0] >= 4.5, ratios)
    low = [f"{x['text']!r} {x['ratio']}" for x in c['low_contrast'] if not x['text'].startswith(FAINT)]
    check(f'{where}: nothing in the dialog falls below 3:1', not low, low[:3])

In [ ]:
passed = sum(1 for r in RESULTS if r['pass'])
total = len(RESULTS)
print(f'\n{passed}/{total} checks passed')

out = SITE / 'tests' / 'results-fhkit-browser.json'
out.write_text(json.dumps({'passed': passed, 'total': total, 'checks': RESULTS},
                          indent=1, ensure_ascii=False), encoding='utf8')
print('written to', out)

assert passed == total, f'{total - passed} check(s) failed'